# Fairness Analysis
Reproduces paper Tables 4 and 6 — fairness metrics and pairwise statistical tests.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from statsmodels.stats.multitest import multipletests

# Paper Table 4 data
TABLE4 = [
    dict(id='A1', name='Basic Aug',  f1=0.506, sens=48.7, spec=60.3, eod_g=12.1, eod_e=16.2, dpd_g=8.5,  dpd_e=11.3, spg=26.4),
    dict(id='A2', name='3D Aug M7', f1=0.946, sens=94.1, spec=95.1, eod_g=2.1,  eod_e=3.9,  dpd_g=1.8,  dpd_e=3.2,  spg=7.3),
    dict(id='A3', name='Std.Prune', f1=0.874, sens=87.4, spec=88.1, eod_g=8.3,  eod_e=11.6, dpd_g=6.1,  dpd_e=8.9,  spg=19.8),
    dict(id='B1', name='PFP',       f1=0.904, sens=91.7, spec=92.3, eod_g=2.1,  eod_e=3.9,  dpd_g=1.8,  dpd_e=3.4,  spg=8.1),
    dict(id='B2', name='INT8',      f1=0.919, sens=92.0, spec=92.6, eod_g=2.4,  eod_e=4.2,  dpd_g=2.0,  dpd_e=3.7,  spg=8.9),
    dict(id='B3', name='KD MNv3',   f1=0.891, sens=89.8, spec=90.5, eod_g=3.1,  eod_e=5.0,  dpd_g=2.6,  dpd_e=4.3,  spg=10.2),
    dict(id='B4', name='PFP+INT8',  f1=0.887, sens=89.2, spec=89.9, eod_g=2.3,  eod_e=4.0,  dpd_g=1.9,  dpd_e=3.5,  spg=8.4),
    dict(id='C1', name='M7+StdPrn', f1=0.921, sens=92.4, spec=93.0, eod_g=7.9,  eod_e=10.8, dpd_g=5.8,  dpd_e=8.4,  spg=17.6),
    dict(id='C2', name='M7+PFP',    f1=0.934, sens=93.5, spec=94.1, eod_g=2.0,  eod_e=3.7,  dpd_g=1.7,  dpd_e=3.1,  spg=7.8),
    dict(id='C3', name='M7+INT8',   f1=0.938, sens=94.0, spec=94.7, eod_g=2.2,  eod_e=3.8,  dpd_g=1.9,  dpd_e=3.3,  spg=7.5),
    dict(id='C4', name='M1+PFP',    f1=0.871, sens=87.5, spec=88.2, eod_g=2.4,  eod_e=4.1,  dpd_g=2.0,  dpd_e=3.6,  spg=8.3),
]

df4 = pd.DataFrame(TABLE4)
TAU_FAIR = 10.0

print('Table 4 — Comprehensive fairness metrics:')
display(df4.style.applymap(
    lambda v: 'background-color: #FFE0E0' if isinstance(v, float) and v > TAU_FAIR else '',
    subset=['eod_g', 'eod_e']
).format(precision=1))

In [ ]:
# Horizontal grouped bar chart — EOD comparison
fig, ax = plt.subplots(figsize=(11, 7))

ids   = df4['id'].tolist()
eod_g = df4['eod_g'].tolist()
eod_e = df4['eod_e'].tolist()

DFZ = {'B1','B2','B3','B4','C2','C3','C4'}
x = np.arange(len(ids))
W = 0.38

def bar_color(cid, val):
    if val > TAU_FAIR: return '#D85A30'   # constraint violation
    if cid == 'C2':    return '#1A6B3A'   # knee point
    if cid in DFZ:     return '#2E8B57'   # DFZ qualified
    return '#888888'                       # not qualified

colors_g = [bar_color(c, v) for c, v in zip(ids, eod_g)]
colors_e = [bar_color(c, v) for c, v in zip(ids, eod_e)]

ax.barh(x + W/2, eod_g, W, color=colors_g, alpha=0.9, label='EOD_gender')
ax.barh(x - W/2, eod_e, W, color=colors_e, alpha=0.55, label='EOD_ethnicity')

ax.axvline(TAU_FAIR, color='red', lw=1.8, ls='--', label=f'τ_fair = {TAU_FAIR}%')
ax.set_yticks(x)
ax.set_yticklabels(ids, fontsize=12)
ax.set_xlabel('EOD (%)', fontsize=12)
ax.set_title('EOD_gender and EOD_ethnicity — all 11 configurations', fontsize=13)
ax.invert_yaxis()

# Highlight C2
c2_idx = ids.index('C2')
ax.axhspan(c2_idx - 0.5, c2_idx + 0.5, alpha=0.07, color='#2E8B57', zorder=0)
ax.text(0.5, c2_idx, '★ knee', fontsize=9, va='center', color='#1A6B3A')

handles = [
    mpatches.Patch(color='#1A6B3A', label='C2: Pareto knee point'),
    mpatches.Patch(color='#2E8B57', label='DFZ-qualified'),
    mpatches.Patch(color='#D85A30', label='EOD constraint violated'),
    mpatches.Patch(color='#888888', label='Not DFZ-qualified'),
    plt.Line2D([0],[0], color='red', lw=1.8, ls='--', label='τ_fair = 10%'),
]
ax.legend(handles=handles, loc='lower right', fontsize=9)

# DFZ/exclusion annotation
for i, (cid, g, e) in enumerate(zip(ids, eod_g, eod_e)):
    mark = '✓ DFZ' if cid in DFZ else '✗'
    ax.text(max(g, e)+0.3, i, mark, va='center', fontsize=8,
            color='#2E8B57' if cid in DFZ else '#D85A30')

plt.tight_layout()
plt.savefig('eod_comparison.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Table 6: Pairwise comparisons (paper values)
TABLE6 = [
    dict(comparison='C2 vs. A3', delta_f1=+0.060, p=0.003,  d=1.42, sig='**',
         interpretation='Fair compression dominates std. pruning on all 3 axes'),
    dict(comparison='C2 vs. B1', delta_f1=+0.030, p=0.021,  d=0.88, sig='*',
         interpretation='3D aug adds marginal gain beyond compress-only'),
    dict(comparison='C2 vs. C4', delta_f1=+0.063, p=0.001,  d=1.78, sig='**',
         interpretation='3D-aware aug is dominant accuracy driver'),
    dict(comparison='B1 vs. A3', delta_f1=+0.030, p=0.018,  d=0.82, sig='*',
         interpretation='PFP strictly Pareto-dominates std. pruning at equal size'),
    dict(comparison='C2-ATWS vs C2-fixed', delta_f1=+0.013, p=0.041, d=0.51, sig='*',
         interpretation='ATWS provides consistent marginal gain'),
]

df6 = pd.DataFrame(TABLE6)
print('Table 6 — Pairwise comparisons (Wilcoxon, Holm-Bonferroni corrected):')
display(df6)

In [ ]:
# Table 7: ATWS ablation
TABLE7 = [
    dict(config='B1', training='Fixed (0.4,0.4,0.2)', f1=0.891, sens=89.6, eod_g=2.8, eod_e=4.6, dpd_g=2.3, dpd_e=4.0),
    dict(config='B1', training='ATWS',                f1=0.904, sens=91.7, eod_g=2.1, eod_e=3.9, dpd_g=1.8, dpd_e=3.4),
    dict(config='C2', training='Fixed (0.4,0.4,0.2)', f1=0.921, sens=92.8, eod_g=2.6, eod_e=4.3, dpd_g=2.2, dpd_e=3.8),
    dict(config='C2', training='ATWS',                f1=0.934, sens=93.5, eod_g=2.0, eod_e=3.7, dpd_g=1.7, dpd_e=3.1),
]

df7 = pd.DataFrame(TABLE7)
print('Table 7 — ATWS ablation:')
display(df7)

# Compute deltas
for cid in ['B1','C2']:
    fixed = df7[(df7.config==cid) & (df7.training.str.startswith('Fixed'))].iloc[0]
    atws  = df7[(df7.config==cid) & (df7.training=='ATWS')].iloc[0]
    print(f'{cid} Δ(ATWS-fixed): F1={atws.f1-fixed.f1:+.3f}, '
          f'EOD_g={atws.eod_g-fixed.eod_g:+.1f}%, EOD_e={atws.eod_e-fixed.eod_e:+.1f}%')